# MCTGCL Model Familiarization

In this notebook, we will explore the MCTGCL model and import the repository from GitHub and run the model on the datasets.

Sources

[1] [MCTGCL Github Link](https://github.com/B-Xi/TGRS_2025_MCTGCL/tree/main)


## Github Clonning

1. First we will create or go to the folder where we want to clone the repository.
```bash
cd /home/edwinacevedo/projects/grenoble2/models
```
2. Then we need to clone the repository from GitHub to our account.
```bash
git clone https://github.com/B-Xi/TGRS_2025_MCTGCL.git
cd TGRS_2025_MCTGCL
```
2. Then we need to install the dependencies.
```bash
conda create -n mctgcl -c pytorch -c nvidia -c conda-forge python=3.10 pytorch torchvision torchaudio pytorch-cuda=12.1 scipy scikit-learn numpy matplotlib einops tqdm onnx -y
conda activate mctgcl
pip install tensorrt pycuda onnxruntime-gpu
```

## For First Run MCTGCL

1. You must creare a data folder in the root of the project.

```bash
mkdir data
cd data
```

2. You must download, copy or move the datasets for train and test (Pavia University by default).

```bash
# For download
wget http://www.ehu.eus/ccwintco/uploads/e/ee/PaviaU.mat
wget http://www.ehu.eus/ccwintco/uploads/5/50/PaviaU_gt.mat

# For copy (Pavia University)
cp ~/datasets/hsi/PaviaU.mat data/paviaU.mat
cp ~/datasets/hsi/PaviaU_gt.mat data/paviaU_gt.mat

# For copy (Indian Pines)
cp ~/datasets/hsi/Indian_pines_corrected.mat data/Indian.mat 
cp ~/datasets/hsi/Indian_pines_gt.mat data/Indian_gt.mat

# For move use mv instead of cp

```

3. You must create a folder for parameters and results of the training stage.

```bash
mkdir results params
```

### Changes in `train.py` code

1. Line 11 change (correction to a local importation)
```python
#import MCTGCL.mctgcl as mctgcl
import mctgcl
```

2. Line 20 change (replace 4 by 2)

```python
#num_tokens = (patch_size - 4) ** 2 
num_tokens = (patch_size - 2) ** 2
```

3. Line 29-30 change (remove `./` from path)

```python
#data_path = os.path.join(os.getcwd(),'./data')
data_path = os.path.join(os.getcwd(),'data')
```

4. Line 165 change (Adjust GPU device if needed)

```python
#device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

5. Line 167 change (replace `massformer` by `mctgcl`)

```python
#net = mctgcl.massformer(num_classes=num_classes, num_tokens=num_tokens).to(device)
net = mctgcl.mctgcl(num_classes=num_classes, num_tokens=num_tokens).to(device)
```

6. Line 176 and 195 change (add a `_` before `net`)

```python
#pred = net(data)
pred, _ = net(data)
```

7. Line 251 change (add `return oa`)

```python
    x_file.write('{}'.format(confusion))
return oa
```

8. Line 259 change (Improve model saving to match `test.py`)

```python
#torch.save(net.state_dict(), 'params/model.pth')
if dataset == 'PU':
    torch.save(net.state_dict(), 'params/Pavia.pth')
elif dataset == 'Indian':
    torch.save(net.state_dict(), 'params/Indian.pth')
```

9. Line 268-269 Change (insert `oa` before `save_reports` and in `get_cls_map`)

```python 
#save_reports(train_time, test_time)
#get_cls_map.get_cls_map(net, device, all_data_loader, y_all)
oa = save_reports(train_time, test_time)
get_cls_map.get_cls_map(net, device, all_data_loader, y_all, oa)
``` 

### Changes in `test.py` code

1. Line 10 change (correction to a local importation)
```python
#import MCTGCL.mctgcl as mctgcl
import mctgcl
```

2. Line 15-16 change (remove `./` from path)

```python
#data_path = os.path.join(os.getcwd(),'./data')
data_path = os.path.join(os.getcwd(),'data')
```

3. Line 21 change (replace 4 by 2)

```python
#num_tokens = (patch_size - 4) ** 2 
num_tokens = (patch_size - 2) ** 2
```
4. Line 164 change (add a `_` before `net`)

```python
#pred = net(data)
pred, _ = net(data)
```

5. Line 210 change (Adjust GPU device if needed)

```python
#device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

6. Line 212 change (replace `massformer` by `mctgcl`)
```python
#net = mctgcl.massformer(num_classes=num_classes, num_tokens=num_tokens).to(device)
net = mctgcl.mctgcl(num_classes=num_classes, num_tokens=num_tokens).to(device)
```

7. Line 215 change (remove `./` from path)

```python
#net.load_state_dict(torch.load('./params/Pavia.pth'))
net.load_state_dict(torch.load('params/Pavia.pth'))
```


## Exporting Model

Once you has already trained the model you can export it to ONNX and TensorRT format.

### Changes in `mctgcl.py` file (due in `mctgcl_onnx.py`)

1. Line 12-14 change (deleting pool2D due to ONNX doesn't support Adaptative Pooling when output size is dinamyc) - ONNX Fix

```python
#self.agp = nn.AdaptiveAvgPool2d((1, 1))
#self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
#self.pool_w = nn.AdaptiveAvgPool2d((1, None))
```

2. Line 25-26 change (changing the pool to include the mean replacing operation of functions deleted above) - ONNX Fix

```python
#x_h = self.pool_h(group_x)
#x_w = self.pool_w(group_x).permute(0, 1, 3, 2)
x_h = group_x.mean(dim=3, keepdim=True) # pool_h equivalent
x_w = group_x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2) # pool_w equivalent
```

3. Line 33-35 change (adding the mean to the operation) - ONNX Fix

```python
#x_hw=self.agp(x2)
####
#x11 = self.softmax((self.agp(x1)+x_hw).reshape(b * self.groups, -1, 1).permute(0, 2, 1))
x_hw = x2.mean(dim=(2, 3), keepdim=True) 
x1_agp = x1.mean(dim=(2, 3), keepdim=True)
x11 = self.softmax((x1_agp + x_hw).reshape(b * self.groups, -1, 1).permute(0, 2, 1))
```

4. Line 282-284 change (TensorRT strictly enforces that 2D pooling operations must receive a 4D tensor) - TensorRT fix

```python
#max_token= self.maxpool(xconv11)
##print(max_token.shape)
#avg_token = self.avgpool(xconv11)

# Add a dummy channel dimension to make it explicitly 4D -> (Batch, 1, H*W, C)
xconv11_4d = xconv11.unsqueeze(1)
# Apply pooling on the 4D tensor, then remove the dummy dimension to revert to 3D
max_token = self.maxpool(xconv11_4d).squeeze(1)
avg_token = self.avgpool(xconv11_4d).squeeze(1)
```


### ONNX Exportation

In [7]:
import torch
import os
from models.TGRS_2025_MCTGCL.mctgcl_onnx import mctgcl

# Define your dataset parameters (Indian Pines example)
dataset = 'Indian'

if dataset == 'Pavia':
    num_classes = 9
elif dataset == 'Indian':
    num_classes = 16
else:
    raise ValueError("Invalid dataset name")

patch_size = 13
num_tokens = (patch_size - 2) ** 2
pca_components = 30

folderpath = os.path.join(os.getcwd(),'models/TGRS_2025_MCTGCL')

# Initialize the model and load the weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = mctgcl(num_classes=num_classes, num_tokens=num_tokens).to(device)

weights_path = f"{folderpath}/params/{dataset}.pt"
model.load_state_dict(torch.load(weights_path, map_location=device))
model.eval() # Evaluation mode, crucial to disable Dropout/BatchNorm layers

# Create a dummy input tensor (Batch, Channels, PCA_components, patch_size, patch_size)
# Batch size of 1 for inference
dummy_input = torch.randn(1, 1, pca_components, patch_size, patch_size).to(device)

# Export to ONNX
onnx_path = f"{folderpath}/params/mctgcl_{dataset}.onnx"
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=12, # Opset version 11 is highly stable for TensorRT
    do_constant_folding=True,
    input_names=['input'],
    output_names=['logits', 'features'],
    # Enable dynamic axes if you plan to change the batch_size during inference
    dynamic_axes={
        'input': {0: 'batch_size'},
        'logits': {0: 'batch_size'},
        'features': {0: 'batch_size'}
    }
)

print(f"Model successfully exported to: {onnx_path}")

/tmp/ipykernel_3955663/941601324.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_location=device))


Model successfully exported to: /home/edwinacevedo/projects/grenoble2/models/TGRS_2025_MCTGCL/params/mctgcl_Indian.onnx


### TensorRT Exportation

In [8]:
import tensorrt as trt
import os

# TensorRT logger setup (required to see errors/warnings)
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

folderpath = os.path.join(os.getcwd(),'models/TGRS_2025_MCTGCL')
models_list = [f for f in os.listdir(f"{folderpath}/params") if f.endswith(".onnx")]

def build_engine(onnx_file_path, engine_file_path, fp16_mode=True):
    # Create the builder and network
    builder = trt.Builder(TRT_LOGGER)
    
    # EXPLICIT_BATCH is mandatory in modern TensorRT versions
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    
    # Create the parser and read the ONNX file
    parser = trt.OnnxParser(network, TRT_LOGGER)
    
    print(f"Reading ONNX file: {onnx_file_path}...")
    with open(onnx_file_path, 'rb') as model:
        if not parser.parse(model.read()):
            print('ERROR: Failed to parse the ONNX file.')
            for error in range(parser.num_errors):
                print(parser.get_error(error))
            return None
            
    print("ONNX successfully loaded. Configuring builder...")
    
    # Configure the builder parameters
    config = builder.create_builder_config()
    
    # Allocate workspace memory for the optimization process (2GB)
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 8 * (1 << 30))
    
    # ---------------------------------------------------------
    # OPTIMIZATION PROFILE: Required for dynamic batch sizes
    # ---------------------------------------------------------
    profile = builder.create_optimization_profile()
    
    # Define the dimensions: (Batch, Channels, PCA_components, Height, Width)
    # Note: Ensure (30, 13, 13) matches your PCA and patch_size configurations
    # Format: (min_shape, opt_shape, max_shape)
    profile.set_shape("input", 
                      (1, 1, 30, 13, 13),       # Minimum possible input (Batch = 1)
                      (512, 1, 30, 13, 13),    # Optimal/Typical input (Batch = 1024)
                      (1024, 1, 30, 13, 13))    # Maximum possible input (Batch = 2048)
    
    config.add_optimization_profile(profile)
    # ---------------------------------------------------------
    
    # Enable FP16 if the GPU supports it and if requested
    if fp16_mode and builder.platform_has_fast_fp16:
        print("Enabling FP16 optimization...")
        config.set_flag(trt.BuilderFlag.FP16)
        
    # Build the engine
    print("Building the TensorRT engine. This may take a couple of minutes...")
    serialized_engine = builder.build_serialized_network(network, config)
    
    if serialized_engine is None:
        print("ERROR: Failed to build the engine.")
        return None
        
    # Save the engine to a file
    with open(engine_file_path, "wb") as f:
        f.write(serialized_engine)
        
    print(f"Engine successfully compiled and saved at: {engine_file_path}!\n")
    return serialized_engine

if __name__ == "__main__":

    for model in models_list:
        onnx_path = f"{folderpath}/params/{model}"
        engine_path = f"{folderpath}/params/{model.split('.')[0]}.engine"
        
        # Compile each engine in the loop
        build_engine(onnx_path, engine_path, fp16_mode=True)

Reading ONNX file: /home/edwinacevedo/projects/grenoble2/models/TGRS_2025_MCTGCL/params/mctgcl_Indian.onnx...
[02/23/2026-06:01:15] [TRT] [W] onnx2trt_utils.cpp:374: Your ONNX model has been generated with INT64 weights, while TensorRT does not natively support INT64. Attempting to cast down to INT32.
ONNX successfully loaded. Configuring builder...
Enabling FP16 optimization...
Building the TensorRT engine. This may take a couple of minutes...
[02/23/2026-06:01:15] [TRT] [W] Detected layernorm nodes in FP16: /transformer/layers.0.1/layernorm/ReduceMean_1, /transformer/layers.1.0/norm/ReduceMean_1, /transformer/layers.0.0/norm/Sub, /transformer/layers.0.0/norm/Pow, /transformer/layers.0.0/norm/ReduceMean_1, /transformer/layers.0.0/norm/Add, /transformer/layers.0.0/norm/Sqrt, /transformer/layers.0.0/norm/Div, /transformer/layers.0.0/norm/Mul, /transformer/layers.0.0/norm/Add_1, /transformer/layers.0.1/layernorm/Sub, /transformer/layers.0.1/layernorm/Pow, /transformer/layers.0.1/layernor

## Fast Testing of Model Formats

Run `inference.py` to evaluate the model on the test set.

### Preliminary Results 

**Overall Accuracy**

TensorRT with FP16 precision instead FP32 of Pytorch ad ONNX model versions.

[Pavia University Dataset]
| Framework | Overall Accuracy |
| --- | --- |
| PyTorch | 96.37% |
| ONNX | 96.37% |
| TensorRT | 96.36% |

[Indian Pines Dataset] <br>
Overal Accuracy in all frameworks is 95.18%.

**Memory Utilization**

TensorRT consumption is highger due to the batch size optimization selected as 1024 as maximum, Pytorch and ONNX request memory to GPU when they know the size of the input data, but TensorRT dont waste time requesting in inference, so at the begining it request more memory to be ready for the maximum batch size of model generated. 

| Framework | Memory Utilization |
| --- | --- |
| PyTorch | 706MiB |
| ONNX | 740MiB |
| TensorRT | 2100MiB  |


**Inference Time vs Batch Size**

TensorRT engine built opimized to 512 batch size, with the following input shape:

```Python 
(1, 1, 30, 13, 13),      # Minimum possible input (Batch = 1)
(512, 1, 30, 13, 13),    # Optimal/Typical input (Batch = 1024)
(1024, 1, 30, 13, 13))   # Maximum possible input (Batch = 2048)
```

| Batch | Pytorch | ONNX | TensorRT |
| --- | --- | --- | --- |
| 1 | 2.5348 ms | 1.2034 ms | 0.5041 ms |
| 8 | 0.3353 ms | 0.1806 ms | 0.1238 ms |
| 16 | 0.1791 ms | 0.1133 ms | 0.1238 ms |
| 32 | 0.0937 ms | 0.0821 ms | 0.1143 ms |
| 128 | 0.0460 ms | 0.0619 ms | 0.0919 ms |
| 512 | 0.0416 ms | 0.0543 ms | 0.0985 ms |

Analysis:
1. **TensorRT dominance in unit inference (Batch 1 - 8):** For individual processing, TensorRT (0.5041 ms) outperforms PyTorch (2.5348 ms) by a factor of ~5x. Being statically pre-compiled at the hardware level, TensorRT eliminates the massive Python validation and dispatch overhead that severely penalizes PyTorch and ONNX when executing small operations iteratively.

2. **Crossover point (Batch 16 - 32):** As the batch size increases, the computational cost of initiating the execution command from Python is diluted across multiple samples, allowing ONNX and PyTorch to match TensorRT's performance.

3. **PyTorch/ONNX scalability in massive batches (Batch 128 - 512):** For large batch sizes, PyTorch and ONNX demonstrate superior efficiency (0.0416 ms and 0.0543 ms, respectively). The stagnation in TensorRT's performance (~0.09 ms) is due to a data transfer bottleneck.

## Extras (Computational Work and Parameters)

Using `thop` library will be calculing Roofline Model for MCTGCL model.

```bash
# Inside Conda environment
pip install thop
```

In [10]:
import torch
from thop import profile, clever_format
# Asegúrate de importar tu modelo correctamente
from models.TGRS_2025_MCTGCL.mctgcl import mctgcl 

def extract_roofline_metrics():
    # Instanciamos el modelo con tu parche P=13 (num_tokens = 11x11 = 121)
    model = mctgcl(num_classes=16, num_tokens=121)
    model.eval()
    
    # Tensor de entrada para 1 solo parche (BS=1)
    inputs = torch.randn(512, 1, 30, 13, 13)
    
    # Extracción de MACs (W) y Parámetros
    macs, params = profile(model, inputs=(inputs, ), verbose=False)
    macs_fmt, params_fmt = clever_format([macs, params], "%.3f")
    
    print("=== Roofline Model Metrics (Per Patch) ===")
    print(f"Work (W) - MACs : {macs} ({macs_fmt})")
    print(f"Parameters      : {params} ({params_fmt})")
    print("==========================================")

if __name__ == '__main__':
    extract_roofline_metrics()

=== Roofline Model Metrics (Per Patch) ===
Work (W) - MACs : 14547214848.0 (14.547G)
Parameters      : 223441.0 (223.441K)
